# DeepSeek-R1-Distill-Qwen 7B — deepseek-ai/DeepSeek-R1-Distill-Qwen-7B

Part of the M.S. thesis *Hate Speech Classification in Roman Urdu: A Comparative Study on Parameter Efficient Fine-Tuning and Prompt Engineering*.

Fine-tunes **DeepSeek-R1-Distill-Qwen 7B** (`deepseek-ai/DeepSeek-R1-Distill-Qwen-7B`) for binary hate speech classification of Roman Urdu comments with **QLoRA**: 4-bit NF4 quantisation (bitsandbytes) plus LoRA adapters (r=16, alpha=8, dropout=0.05) via Hugging Face PEFT. A custom `Trainer` uses class-weighted cross-entropy to handle the 82/18 class imbalance. The notebook first measures the untrained baseline, then trains for 2 epochs and reports accuracy, balanced accuracy and macro precision/recall/F1 on the held-out test set.

**Setup:** install dependencies with `pip install -r requirements.txt` from the repository root and place the dataset in `data/` (see the main README).

**Hardware:** a CUDA GPU with at least 16 GB of memory is recommended.

In [ ]:
import os

# Paths are relative to this notebook. See data/README.md for the dataset.
DATA_PATH = "../../data/PURRUTT.xlsx"
OUTPUT_DIR = "../../outputs/deepseek_r1_distill_qwen_7b_qlora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Weights & Biases logging is off by default; set to "false" to enable it.
os.environ.setdefault("WANDB_DISABLED", "true")
# Gated models (e.g. Llama 3, Gemma) need a Hugging Face token:
# run `huggingface-cli login` or set the HF_TOKEN environment variable before starting.

In [ ]:
import os
import random
import functools
import csv
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import evaluate

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report, balanced_accuracy_score, accuracy_score

from datasets import Dataset, DatasetDict
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
df = pd.read_excel(DATA_PATH)
df

### Train / validation / test split
* Sequential split: first 60% train, next 20% validation, last 20% test.

In [ ]:
train_end_point = int(df.shape[0]*0.6)
val_end_point = int(df.shape[0]*0.8)
df_train = df.iloc[:train_end_point,:]
df_val = df.iloc[train_end_point:val_end_point,:]
df_test = df.iloc[val_end_point:,:]
print(df_train.shape, df_test.shape, df_val.shape)

### Convert from Pandas DataFrame to Hugging Face Dataset
* Also let's shuffle the training set.
* We put the components train,val,test into a DatasetDict so we can access them later with HF trainer.
* Later we will add a tokenized dataset

In [ ]:
# Converting pandas DataFrames into Hugging Face Dataset objects:
dataset_train = Dataset.from_pandas(df_train.drop('Urdu',axis=1))
dataset_val = Dataset.from_pandas(df_val.drop('Urdu',axis=1))
dataset_test = Dataset.from_pandas(df_test.drop('Urdu',axis=1))

In [ ]:
# Shuffle the training dataset
dataset_train_shuffled = dataset_train.shuffle(seed=42)  # Using a seed for reproducibility

In [ ]:
# Combine them into a single DatasetDict
dataset = DatasetDict({
    'train': dataset_train_shuffled,
    'val': dataset_val,
    'test': dataset_test
})
dataset

In [ ]:
dataset['train']

* Since our classes are not balanced let's calculate class weights based on inverse value counts
* Convert to pytorch tensor since we will need it

In [ ]:
df_train.Toxic.value_counts(normalize=True)

In [ ]:
class_weights=(1/df_train.Toxic.value_counts(normalize=True).sort_index()).tolist()
class_weights=torch.tensor(class_weights)
class_weights=class_weights/class_weights.sum()
class_weights

In [ ]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

#### Quantization Config (for QLORA)

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True, # enable 4-bit quantization
    bnb_4bit_quant_type = 'nf4', # information theoretically optimal dtype for normally distributed weights
    bnb_4bit_use_double_quant = True, # quantize quantized weights //insert xzibit meme
    bnb_4bit_compute_dtype = torch.bfloat16 # optimized fp format for ML
)

#### Lora Config

In [ ]:
lora_config = LoraConfig(
    r = 16, # the dimension of the low-rank matrices
    lora_alpha = 8, # scaling factor for LoRA activations vs pre-trained weight activations
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout = 0.05, # dropout probability of the LoRA layers
    bias = 'none', # wether to train bias weights, set to 'none' for attention layers
    task_type = 'SEQ_CLS'
)

#### Load model
* AutomodelForSequenceClassification
* Num Labels is # of classes

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    num_labels=2
)

model

* prepare_model_for_kbit_training() function to preprocess the quantized model for training.

In [ ]:
model = prepare_model_for_kbit_training(model)

* get_peft_model prepares a model for training with a PEFT method such as LoRA by wrapping the base model and PEFT configuration with get_peft_model

In [ ]:
model = get_peft_model(model, lora_config)

### Load the tokenizer
* The base model has no dedicated padding token, so the EOS token is used for padding.

In [ ]:
# ✅ Ensure model_name is a string
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  # Replace if incorrect

# ✅ Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(str(model_name), trust_remote_code=True, use_fast=True)

# ✅ Set padding token (Qwen & Falcon models do not have a pad token by default)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
# ✅ Print mismatch info
print(f"Model Embedding Size: {model.get_input_embeddings().weight.shape[0]}")
print(f"Tokenizer Vocab Size: {len(tokenizer)}")

In [ ]:
# ✅ Resize model embedding layer to match tokenizer
model.resize_token_embeddings(len(tokenizer))

# ✅ Move model to GPU (optional)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✅ New Model Embedding Size: {model.get_input_embeddings().weight.shape[0]}")

#### Update model config
* `use_cache = False` is required for training with gradient checkpointing.

In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.config.pretraining_tp = 1

In [ ]:
sentences = df_test.Comment.tolist()
sentences[0:2]

In [ ]:
from tqdm import tqdm
import torch
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


# ✅ Convert summaries to a list
sentences = df_test.Comment.tolist()

# ✅ Define the batch size
batch_size = 10  # Adjust based on system memory

# ✅ Initialize an empty list to store the model outputs
all_outputs = []

# ✅ Process the sentences in batches with tqdm progress bar
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i in tqdm(range(0, len(sentences), batch_size), desc="Processing Batches", unit="batch"):
    # ✅ Get the batch of sentences
    batch_sentences = sentences[i:i + batch_size]

    # ✅ Tokenize the batch
    # 👇 Reduced max_length to 128 to match the value used during training
    inputs = tokenizer(batch_sentences, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # ✅ Ensure device is properly defined
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ✅ Move inputs safely
    inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}


    # ✅ Perform inference and store the logits
    with torch.no_grad():
        outputs = model(**inputs)
        all_outputs.append(outputs['logits'])

* Concatenate all outputs into a single tensor

In [ ]:
final_outputs = torch.cat(all_outputs, dim=0)
final_outputs

* argmax to get class prediction

In [ ]:
final_outputs.argmax(axis=1)

* Move to CPU so we can use numpy and set prediction colum to it

In [ ]:
df_test['predictions']=final_outputs.argmax(axis=1).cpu().numpy()
df_test['predictions']

In [ ]:
df_test['predictions'].value_counts()

In [ ]:
def get_performance_metrics(df_test):
  y_test = df_test.Toxic
  y_pred = df_test.predictions

  print("Confusion Matrix:")
  print(confusion_matrix(y_test, y_pred))

  print("\nClassification Report:")
  print(classification_report(y_test, y_pred))

  print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred))
  print("Accuracy Score:", accuracy_score(y_test, y_pred))

In [ ]:
get_performance_metrics(df_test)

# Trainer Components
* model
* tokenizer
* training arguments
* train dataset
* eval dataset
* Data Collater
* Compute Metrics
* class_weights: In our case since we are using a custom trainer so we can use a weighted loss we will subclass trainer and define the custom loss.

#### Tokenise the train / validation / test splits

In [ ]:
MAX_LEN = 128
col_to_delete = ['Urdu']

def mistral_preprocessing_function(examples):
    return tokenizer(examples['Comment'], truncation=True, max_length=MAX_LEN)

tokenized_datasets = dataset.map(mistral_preprocessing_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("Toxic", "label")
tokenized_datasets.set_format("torch")

## Data Collator
A **data collator** prepares batches of data for training or inference in machine learning, ensuring uniform formatting and adherence to model input requirements. This is especially crucial for variable-sized inputs like text sequences.

### Functions of Data Collator

1. **Padding:** Uniformly pads sequences to the length of the longest sequence using a special token, allowing simultaneous batch processing.
2. **Batching:** Groups individual data points into batches for efficient processing.
3. **Handling Special Tokens:** Adds necessary special tokens to sequences.
4. **Converting to Tensor:** Transforms data into tensors, the required format for machine learning frameworks.

### `DataCollatorWithPadding`

The `DataCollatorWithPadding` specifically manages padding, using a tokenizer to ensure that all sequences are padded to the same length for consistent model input.

- **Syntax:** `collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)`
- **Purpose:** Automatically pads text data to the longest sequence in a batch, crucial for models like BERT or GPT.
- **Tokenizer:** Uses the provided `tokenizer` for sequence processing, respecting model-specific vocabulary and formatting rules.

This collator is commonly used with libraries like Hugging Face's Transformers, facilitating data preprocessing for various NLP models.

In [ ]:
collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)

# define which metrics to compute for evaluation
* We will use balanced accuracy and accuracy for simplicity

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {'balanced_accuracy' : balanced_accuracy_score(predictions, labels),'accuracy':accuracy_score(predictions,labels)}

### Define custom trainer with classweights
* We will have a custom loss function that deals with the class weights and have class weights as additional argument in constructor

In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Ensure label_weights is a tensor
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32).to(self.args.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False):
        # Extract labels and convert them to long type for cross_entropy
        labels = inputs.pop("labels").long()

        # Forward pass
        outputs = model(**inputs)

        # Extract logits assuming they are directly outputted by the model
        logits = outputs.get('logits')

        # Compute custom loss with class weights for imbalanced data handling
        if self.class_weights is not None:
            loss = F.cross_entropy(logits, labels, weight=self.class_weights)
        else:
            loss = F.cross_entropy(logits, labels)

        return (loss, outputs) if return_outputs else loss

# define training args

In [ ]:
training_args = TrainingArguments(
    output_dir = os.path.join(OUTPUT_DIR, "checkpoints"),
    learning_rate = 1e-4,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    num_train_epochs = 2,
    weight_decay = 0.01,
    evaluation_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True
)

#### Define custom trainer

In [ ]:
trainer = CustomTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_datasets['train'],
    eval_dataset = tokenized_datasets['val'],
    tokenizer = tokenizer,
    data_collator = collate_fn,
    compute_metrics = compute_metrics,
    class_weights=class_weights,
)

* https://huggingface.co/docs/transformers/en/training

### Run trainer!

In [ ]:
train_result = trainer.train()

#### Evaluate the fine-tuned model on the test set

In [ ]:
from tqdm import tqdm
import torch

def make_predictions(model, df_test):
    # Convert summaries to a list
    sentences = df_test.Comment.tolist()

    # Define the batch size
    batch_size = 32  # You can adjust this based on your system's memory capacity

    # Initialize an empty list to store the model outputs
    all_outputs = []

    # Process the sentences in batches with tqdm progress bar
    for i in tqdm(range(0, len(sentences), batch_size), desc="Processing Batches", unit="batch"):
        # Get the batch of sentences
        batch_sentences = sentences[i:i + batch_size]

        # Tokenize the batch
        inputs = tokenizer(batch_sentences, return_tensors="pt", padding=True, truncation=True, max_length=128)

        # Move tensors to the device where the model is (e.g., GPU or CPU)
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Perform inference and store the logits
        with torch.no_grad():
            outputs = model(**inputs)
            all_outputs.append(outputs['logits'])

    # Concatenate all outputs
    final_outputs = torch.cat(all_outputs, dim=0)

    # Convert logits to class predictions
    df_test['predictions'] = final_outputs.argmax(axis=1).cpu().numpy()

    # Optional: Map predictions to categories if needed
    # df_test['predictions'] = df_test['predictions'].apply(lambda l: category_map[l])

    print("Prediction process completed!")

make_predictions(model, df_test)

In [ ]:
def get_performance_metrics(df_test):
  y_test = df_test.Toxic
  y_pred = df_test.predictions

  print("Confusion Matrix:")
  print(confusion_matrix(y_test, y_pred))

  print("\nClassification Report:")
  print(classification_report(y_test, y_pred))

  print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred))
  print("Accuracy Score:", accuracy_score(y_test, y_pred))

In [ ]:
get_performance_metrics(df_test)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def calculate_macro_scores(y_true, y_pred):
    y_test = df_test.Toxic
    y_pred = df_test.predictions
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')

    print(f"Macro Precision: {precision:.4f}")
    print(f"Macro Recall: {recall:.4f}")
    print(f"Macro F1-score: {f1:.4f}")

    return precision, recall, f1


y_test = df_test.Toxic
y_pred = df_test.predictions
calculate_macro_scores(y_test, y_pred)

### Saving the model trainer state and model adapters

In [ ]:
metrics = train_result.metrics
max_train_samples = len(dataset_train)
metrics["train_samples"] = min(max_train_samples, len(dataset_train))
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

#### Saving the adapter model
* Note this doesn't save the entire model. It only saves the adapters.

In [ ]:
trainer.save_model(os.path.join(OUTPUT_DIR, "adapter"))